<a href="https://colab.research.google.com/github/wellingtonsjbdev/projeto-drosophiala/blob/main/cerebro-de-mosca-reservoir.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install neuprint-python

Para começar a entender como o cérebro da mosca funciona como uma 'IA', o primeiro passo é obter dados sobre as **conexões sinápticas** entre os neurônios. Isso nos permite ver como os neurônios se comunicam.

In [ ]:
from neuprint import Client
from google.colab import userdata

token = userdata.get("NEUPRINT_TOKEN")

# Stripping whitespace and potential newline characters from the token
# This also prevents the token from being exposed in error messages due to malformed headers.
if token:
    token = token.strip()
else:
    raise ValueError("Configure NEUPRINT_TOKEN nos segredos do Colab.")

c = Client(
    "neuprint.janelia.org",
    dataset="male-cns:v1.0",
    token=token
)

In [ ]:
# Vamos buscar informações sobre as conexões sinápticas entre neurônios.
# Por exemplo, podemos buscar os 100 primeiros pares de neurônios conectados (pre-synaptic e post-synaptic).
# Note que 'FROM' e 'TO' representam os bodyIds dos neurônios.
# 'weight' representa o número de sinapses entre eles.

connections_df = c.fetch_custom(
    "MATCH (pre:Neuron)-[r:ConnectsTo]->(post:Neuron) RETURN pre.bodyId AS pre_bodyId, post.bodyId AS post_bodyId, r.weight AS weight LIMIT 100"
)
display(connections_df.head())


Com esses dados de conectividade, podemos começar a pensar em como aplicar conceitos de redes neurais ou análise de grafos para simular ou entender aspectos da "IA" do cérebro da mosca.

**Reasoning**:
To identify 1000 target neuron body IDs, I will execute a Cypher query using the `c.fetch_custom` method to retrieve the bodyId of 1000 neurons and store the result in a DataFrame called `target_neuron_body_ids`.



In [ ]:
target_neuron_body_ids = c.fetch_custom("MATCH (n:Neuron) RETURN n.bodyId LIMIT 1000")
display(target_neuron_body_ids.head())

**Reasoning**:
The first step is to extract the body IDs from the `target_neuron_body_ids` DataFrame into a list, which will be used to filter subsequent Cypher queries for detailed neuron information and connections.



In [ ]:
target_neuron_ids = target_neuron_body_ids['n.bodyId'].tolist()
print(f"Extracted {len(target_neuron_ids)} target neuron IDs.")

**Reasoning**:
Now that the target neuron IDs are extracted, I will fetch detailed information for these 1000 neurons (ID, type, instance) and all their synaptic connections (both pre- and post-synaptic) using `c.fetch_custom` and then display the head of both resulting DataFrames.



In [ ]:
neuron_details_df = c.fetch_custom(
    f"MATCH (n:Neuron) WHERE n.bodyId IN {list(target_neuron_ids)} RETURN n.bodyId, n.type, n.instance"
)
display(neuron_details_df.head())

# Modificado para buscar conexões para apenas os primeiros 100 neurônios alvo para evitar travamento.
all_connections_df = c.fetch_custom(
    f"MATCH (pre:Neuron)-[r:ConnectsTo]->(post:Neuron) WHERE pre.bodyId IN {list(target_neuron_ids[:100])} OR post.bodyId IN {list(target_neuron_ids[:100])} RETURN pre.bodyId AS pre_bodyId, post.bodyId AS post_bodyId, r.weight AS weight"
)
display(all_connections_df.head())

## Construir um Grafo de Conectividade

### Subtask:
Utilize a biblioteca NetworkX para construir um grafo a partir dos dados de conexão sináptica (`all_connections_df`). Os neurônios (`neuron_details_df`) serão os nós e as conexões serão as arestas, com os pesos representando a força sináptica. Isso é essencial para visualizar a arquitetura da rede neural e prepará-la para análises mais aprofundadas.

In [ ]:
import networkx as nx

# Crie um grafo direcionado
G = nx.DiGraph()

# Adicione os neurônios como nós ao grafo
# Usamos o 'bodyId' como identificador único para cada nó
# Adicionamos atributos como 'type' e 'instance' se disponíveis em neuron_details_df
for index, row in neuron_details_df.iterrows():
    G.add_node(row['n.bodyId'], type=row['n.type'], instance=row['n.instance'])

# Adicione as conexões sinápticas como arestas ao grafo
# Usamos 'pre_bodyId' como nó de origem, 'post_bodyId' como nó de destino e 'weight' como atributo da aresta
for index, row in all_connections_df.iterrows():
    G.add_edge(row['pre_bodyId'], row['post_bodyId'], weight=row['weight'])

print(f"Grafo criado com {G.number_of_nodes()} nós e {G.number_of_edges()} arestas.")

# Exiba algumas informações básicas sobre o grafo
print("Alguns nós no grafo:", list(G.nodes())[:5])
print("Algumas arestas no grafo:", list(G.edges(data=True))[:5])

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

# --- 1. Visualização Avançada do Cérebro ---
plt.figure(figsize=(10, 10), facecolor='#111111')
ax = plt.gca()
ax.set_facecolor('#111111')

# Filtrando os 100 neurônios mais conectados
grau = dict(G.degree())
top_100 = sorted(grau, key=grau.get, reverse=True)[:100]
subG = G.subgraph(top_100)

# Layout espacial para melhor distribuição visual
pos = nx.spring_layout(subG, k=0.15, seed=42)

# Configurações estéticas baseadas no grau (grau de conexões define o tamanho e cor do nó)
node_degrees = [grau[node] for node in subG.nodes()]

# Desenhar conexões (arestas) com cor compatível com Matplotlib
nx.draw_networkx_edges(
    subG, pos,
    edge_color='#444444',
    alpha=0.2,
    arrows=True,
    arrowsize=8
)

# Desenhar neurônios (nós)
sc = nx.draw_networkx_nodes(
    subG, pos,
    node_color=node_degrees,
    cmap=plt.cm.plasma,
    node_size=[d * 1.5 for d in node_degrees],
    alpha=0.85
)

# Adicionar barra de cores para indicar conectividade
cbar = plt.colorbar(sc, orientation='horizontal', pad=0.05, shrink=0.6)
cbar.set_label('Intensidade de Conectividade (Grau do Neurônio)', color='white')
cbar.ax.xaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar.ax.axes, 'xticklabels'), color='white')

plt.title("Rede de Conectividade Neural (Top 100 Neurônios)", color='white', fontsize=14, pad=20)
plt.axis('off')
plt.show()

# --- 2. Simulação de Dinâmica de Rede (Reservoir Computing) ---
nodes = list(subG.nodes())
W = nx.to_numpy_array(subG)

# Normalização simples para estabilizar a dinâmica caótica
if W.max() > 0:
    W = W / W.max()

# Simulação dos estados neurais
estado = np.random.rand(100)
historico_estados = [estado]

for t in range(10):
    # Atualização dinâmica de estado usando tangente hiperbólica como função de ativação
    estado = np.tanh(W @ estado * 0.15)
    historico_estados.append(estado)

print(f"-> Depois de 10 passos, ativação média do sistema: {estado.mean():.4f}")
print("-> Sistema dinâmico estabilizado com sucesso. Pronto para análise de séries temporais!")